> 在线讲义：[第 23 章 · 滑动窗口：每层只看前 W 个词，叠起来为什么能看得很远](https://wikieden.github.io/beyond-transformer/lessons/23-sliding-window.html) · 本书《超越 Transformer》

# 第 23 章 · 滑动窗口：每层只看前 W 个词，叠起来为什么能看得很远

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/23-sliding-window.html) · 仓库里 `beyond-transformer/notes/lessons/23-sliding-window.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 感受野 | mask 连乘 | `reach(40, 4, k)` | 3、6、9、15 |
| 第二步 rolling buffer | i mod W | `rolling_buffer(toks, 4)` | ['the', 'mat', 'and', 'on'] |
| 第三步 交替 | 每层存多少 | `kv_tokens(8192, "LG", 4096, 42)` | 258,048（75%） |
| 第四步 soft-cap | cap·tanh | `softcap(xs, 50)` | 1000 → 50.0 |

local / global 交替；logit soft-capping：cap·tanh(x / cap)。
在 a1 模型上加可配置的 SWA 交替模式、soft-cap 是里程碑 T-M5。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `window_mask`**：位置 i 能看 j：i − w < j ≤ i（包括自己，一共 w 个）。

In [ ]:
def window_mask(n, w):
    """位置 i 能看 j：i − w < j ≤ i（包括自己，一共 w 个）。"""
    i, j = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    return ((j <= i) & (j > i - w)).astype(int)

**函数 `reach`**：叠 layers 层后，最后一个位置最远能间接看到多早的位置（布尔矩阵连乘）。

In [ ]:
def reach(n, w, layers):
    """叠 layers 层后，最后一个位置最远能间接看到多早的位置（布尔矩阵连乘）。"""
    m = window_mask(n, w)
    r = np.eye(n, dtype=int)
    for _ in range(layers):
        r = ((m @ r) > 0).astype(int)
    first = int(np.argmax(r[-1] > 0))
    return n - 1 - first

**函数 `rolling_buffer`**：把第 i 个 token 写到 cache 的 i mod w 号槽位，返回最后的 cache 内容。

In [ ]:
def rolling_buffer(tokens, w):
    """把第 i 个 token 写到 cache 的 i mod w 号槽位，返回最后的 cache 内容。"""
    cache = [None] * w
    for i, t in enumerate(tokens):
        cache[i % w] = t
    return cache

**函数 `softcap`**：小工具。

In [ ]:
def softcap(x, cap):
    return cap * np.tanh(np.asarray(x, float) / cap)

**函数 `kv_tokens`**：每层要存多少个 token 的 K、V：local 层最多 w_local 个，global 层 n 个。pattern 例如 'LG'（交替）。

In [ ]:
def kv_tokens(n, pattern, w_local, layers):
    """每层要存多少个 token 的 K、V：local 层最多 w_local 个，global 层 n 个。pattern 例如 'LG'（交替）。"""
    total = 0
    for l in range(layers):
        total += min(n, w_local) if pattern[l % len(pattern)] == "L" else n
    return total

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 感受野随层数线性变宽（窗口 W = 4，含自己）")
for k in (1, 2, 3, 5):
    print(f"  {k} 层：最后一个位置最远能看到 {reach(40, 4, k)} 个位置之前（≈ (W − 1)·k）")
print(f"  Mistral 7B：W = 4096、32 层 → 理论跨度约 4096 × 32 = {4096 * 32:,}")
print("\n第二步 rolling buffer cache（W = 4）")
toks = ["The", "cat", "sat", "on", "the", "mat", "and"]
print(f"  依次写入 {toks}：cache = {rolling_buffer(toks, 4)}（第 i 个存在 i mod 4 号槽）")

In [ ]:
print("\n第三步 local / global 交替省多少 cache（42 层、读到 8192 个 token）")
for name, pat in (("全 global", "G"), ("Gemma 2：local 4096 / global 交替", "LG"), ("5 local : 1 global", "LLLLLG")):
    t = kv_tokens(8192, pat, 4096, 42)
    print(f"  {name}：所有层共存 {t:,} 个 token 的 K、V（全 global 的 {t / (8192 * 42):.0%}）")

In [ ]:
print("\n第四步 logit soft-capping")
xs = [5.0, 20.0, 50.0, 100.0, 1000.0]
print(f"  cap = 50（注意力）：{np.round(softcap(xs, 50), 2).tolist()}")
print(f"  cap = 30（最后的输出层）：{np.round(softcap(xs, 30), 2).tolist()}")

## 3. 代码里才看得到的

- **“窗口含不含自己”会差一格**：这里窗口 W 个位置里包括自己，所以每层多 W − 1 格；Mistral 原文的 W × k 是不细抠这一格的说法。
- **local 层的 cache 也要配合 rolling buffer**：`kv_tokens` 只是数个数；真实实现里 local 层用 W 个槽循环、global 层存全部，两种 cache 要分开管理。

## 4. 自己改一改

- 给 `reach` 换成每隔一层用 global（窗口 = 全长），3 层后最远能看到多远？
- 把 `kv_tokens` 的序列长度改成 131,072，交替模式能省到多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 传话游戏：每层只听近处 W 个，叠 k 层间接听到 W 乘 k；越远越模糊。
> 滚动缓存开 W 个槽，第 i 个放 i 模 W，旧的被盖掉。
> 局部层省、全局层看全，交替着用；cap 乘 tanh 压住大分数，注意力五十、输出三十。